# Content-based recommender – verwerkte feedback

In deze notebook heb ik de feedback van de docent verwerkt voor mijn onderdeel: de **content-based** aanbevelingen. Bij elk onderdeel staat kort:

- **Feedback:** wat de docent opmerkte;
- **Toegepast:** wat ik daarmee heb gedaan en waarom.

De notebook leest de opgeschoonde CSV's uit de eerdere notebooks (`cleaned_Leesmateriaal.csv`, `cleaned_Interacties.csv`, `cleaned_Student.csv`). De codefragmenten uit de feedback waren niet uitgevoerd op onze data. Ik heb ze dus aangepast, samengevoegd met mijn bestaande code en zelf getest.

| # | Onderdeel | Feedback in één zin | Wat ik gedaan heb |
|---|-----------|--------------------|-------------------|
| 1 | Data en datakwaliteit | Controleer sleutels, bereik en dubbele paren | Checks toegevoegd, herhaalde paren bewust geaggregeerd |
| 2 | Feature engineering | Niet alles in één tekst; onderwerp dubbel; schattingen markeren | Vier tekstvarianten, niveau/formaat/uitgever/auteur apart |
| 3 | NLP | Eén functie voor catalogus én profiel | `preprocess()` voor beide, vectorizer opnieuw gefit |
| 4 | Splitsing en labels | Eén vaste split, vaste labels, geen lekkage | Ontwikkeling / validatie / eindtest met beoordeling ≥ 4 |
| 5 | Personalisatie en filters | Onderscheid eisen en voorkeuren; randgevallen | Niveau/tijd/taal als harde eis, genre/type als bonus |
| 6 | Evaluatie | Eén protocol, twee methodes apart, baselines | Gedeelde metricfunctie, candidate én full-catalog, random + populariteit |
| 7 | Tuning en eindtest | Tunen op validatie, eindtest één keer | Kleine grid op validatie, daarna één keer op de eindtest |
| 8 | Eindprototype | Eén gebruikersfunctie met uitleg en tests | `geef_aanbevelingen()` met uitleg en randgeval-tests |
| 9 | Beperkingen | Eerlijk rapporteren | Sectie met beperkingen en conclusie |

**Wat hier niet in zit:** user-based en item-based CF, de hybrid en de dimensiereductie-feedback. Die horen bij andere onderdelen. De evaluatiefuncties zijn zo gebouwd dat de CF-modellen er later onder dezelfde split en metrics in kunnen worden gehangen.

## 1. Data laden en datakwaliteit

**Feedback:** controleer sleutels, het bereik van het leespercentage en dubbele student-materiaalparen. Herhaalde interacties kunnen legitiem zijn, maar aggregeer ze bewust vóór het splitsen.

**Toegepast:** ik laad de drie bestanden en voer de checks uit de feedback uit. De herhaalde paren laat ik bestaan in de ruwe data. Ze worden pas per student-materiaalpaar geaggregeerd (gemiddelde) bij het maken van de labels (sectie 4) en van de interactiescore. Zo gebeurt het vóór het splitsen en op één plek. Rijen met een onbekende student of onbekend materiaal laat ik weg en ik meld hoeveel dat er zijn.

In [ ]:
import re
import warnings
from functools import lru_cache

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

pd.set_option("display.max_colwidth", 90)
pd.set_option("display.width", 160)

PAD_LEES  = "../dataset/cleaned_Leesmateriaal.csv"
PAD_INTER = "../dataset/cleaned_Interacties.csv"
PAD_STUD  = "../dataset/cleaned_Student.csv"

df        = pd.read_csv(PAD_LEES)
raw       = pd.read_csv(PAD_INTER)
studenten = pd.read_csv(PAD_STUD)

# sleutels altijd als tekst, zodat merges en vergelijkingen overal hetzelfde werken
df["cataloguscode"]   = df["cataloguscode"].astype(str)
raw["cataloguscode"]  = raw["cataloguscode"].astype(str)
raw["student_id"]     = raw["student_id"].astype(str)
studenten["student_id"] = studenten["student_id"].astype(str)

print("Materialen:", df.shape, "| Interacties:", raw.shape, "| Studenten:", studenten.shape)

In [ ]:
# --- Checks uit de feedback ---
assert df["cataloguscode"].is_unique, "cataloguscode is niet uniek"
assert studenten["student_id"].is_unique, "student_id is niet uniek"

onbekende_materialen = (~raw["cataloguscode"].isin(df["cataloguscode"])).sum()
onbekende_studenten  = (~raw["student_id"].isin(studenten["student_id"])).sum()
herhaalde_paren      = raw.duplicated(["student_id", "cataloguscode"]).sum()

print("Onbekende materialen in interacties:", onbekende_materialen)
print("Onbekende studenten in interacties :", onbekende_studenten)
print("Herhaalde student-materiaalparen   :", herhaalde_paren, "(worden geaggregeerd, zie sectie 4)")
assert raw["leespercentage"].dropna().between(0, 100).all(), "leespercentage buiten 0-100"
print("Missende beoordelingen             :", f"{raw['beoordeling'].isna().mean():.1%}")

# Afwijkingen: eigenlijk bij de bron herstellen; hier laat ik die rijen weg.
voor = len(raw)
raw = raw[raw["cataloguscode"].isin(df["cataloguscode"]) & raw["student_id"].isin(studenten["student_id"])].copy()
print("Rijen weggelaten door onbekende sleutels:", voor - len(raw))

# Sparsity ter context (zie ook de CF-feedback: ~93% ontbrekend)
n_paren = raw[["student_id", "cataloguscode"]].drop_duplicates().shape[0]
print(f"Dekking student x materiaal: {n_paren / (studenten.shape[0] * df.shape[0]):.1%}")

## 2. Feature engineering: welke tekst gaat de TF-IDF in?

**Feedback:**
- Bewaar niveau, formaat en uitgever afzonderlijk; ze horen niet in de inhoudelijke tekst.
- Een logisch startpunt voor inhoudelijke similarity is omschrijving, onderwerp, trefwoorden en eventueel genre.
- Auteur kan een aparte voorkeur zijn en hoort dus niet in de tekst.
- Onderwerp zit al in trefwoorden: onderzoek de dubbele weging.
- Markeer vóór de imputatie welke leestijden en pagina-aantallen geschat zijn.
- Vergelijk de nieuwe tekstselectie met de bestaande variant, onder hetzelfde evaluatieprotocol.

**Toegepast:**
- Mijn oude `Gecombineerd` bevatte acht kolommen, inclusief Niveau, uitgever, formaat en Auteur. Die variant bewaar ik als referentie (`oud`).
- Daarnaast bouw ik drie varianten volgens de feedback: `fb` (omschrijving + genre + trefwoorden), `fb_zonder_genre` (omschrijving + trefwoorden) en `fb_plus_onderwerp` (zoals `fb`, met onderwerp erbij, om de dubbele weging te testen).
- Welke het beste werkt bepaal ik in sectie 7, op de validatieset.
- Niveau, formaat, uitgever en auteur blijven als aparte kolommen in `df` staan en worden niet meer in de tekst gebruikt.

**Let op, in de opschoon-notebook:** de markering van geschatte waarden moet *vóór* de imputatiecel staan. Voeg daar dit toe en exporteer `cleaned_Leesmateriaal.csv` opnieuw:

```python
# direct VOOR de imputatie van aantal_paginas / geschatte_leestijd_min
df["pagina_geschat"]   = df["aantal_paginas"].isna() & df["geschatte_leestijd_min"].notna()
df["leestijd_geschat"] = df["geschatte_leestijd_min"].isna() & df["aantal_paginas"].notna()
```

In [ ]:
# Markering van geschatte leestijd (moet uit de opschoon-notebook komen)
if "leestijd_geschat" not in df.columns:
    warnings.warn("Kolom 'leestijd_geschat' ontbreekt: voeg de markering toe in de opschoon-notebook "
                  "(zie hierboven). Tot die tijd kan ik niet aangeven welke leestijden geschat zijn.")
    df["leestijd_geschat"] = np.nan
else:
    print("Geschatte leestijden:", int(df["leestijd_geschat"].fillna(False).astype(bool).sum()))

# Tekstvarianten (kolommen die samen de inhoudelijke tekst vormen)
VARIANTEN = {
    "oud":               ["Korte omschrijving", "genre", "onderwerp", "trefwoorden",
                          "Niveau", "uitgever", "beschikbaar_formaat", "Auteur"],
    "fb":                ["Korte omschrijving", "genre", "trefwoorden"],
    "fb_zonder_genre":   ["Korte omschrijving", "trefwoorden"],
    "fb_plus_onderwerp": ["Korte omschrijving", "genre", "onderwerp", "trefwoorden"],
}

def maak_tekst(frame, kolommen):
    return frame[kolommen].fillna("").astype(str).agg(" ".join, axis=1)

for naam, kolommen in VARIANTEN.items():
    df[f"tekst_{naam}"] = maak_tekst(df, kolommen)

# Controle: hoe vaak zit onderwerp al in trefwoorden? (dubbele weging)
onderwerp = df["onderwerp"].fillna("").str.strip().str.casefold()
trefw = df["trefwoorden"].fillna("").str.casefold().str.split(";")
zit_erin = [bool(o) and o in [w.strip() for w in t] for o, t in zip(onderwerp, trefw)]
print(f"Onderwerp komt al voor in trefwoorden bij {np.mean(zit_erin):.0%} van de materialen")

df[["Titel", "tekst_fb"]].head(3)

## 3. Eén NLP-functie voor catalogus én studentprofiel

**Feedback:** de catalogus en de profieltekst doorliepen niet dezelfde stappen. Gebruik één functie voor beide en vervang leestekens door spaties, zodat samengeplakte woorden niet ontstaan. Gebruik `preprocess` ook in de aanbevelingsfunctie en fit de vectorizer opnieuw op de aangepaste catalogus.

**Toegepast:**
- Ik heb `preprocess()` uit de feedback overgenomen: kleine letters, leestekens en losse cijfers naar spaties, stopwoorden eruit, daarna lemmatiseren met spaCy.
- Deze functie gebruik ik voor *alle* tekstvarianten van de catalogus en voor de profieltekst van een student.
- De uitkomst is cached met `lru_cache`, omdat dezelfde tekst vaak terugkomt. Dat scheelt veel tijd bij de evaluatie.
- Elke tekstvariant krijgt zijn eigen TF-IDF-vectorizer. Die wordt aangemaakt in het model in sectie 5, dus de vectorizer is steeds opnieuw op de juiste catalogustekst gefit.
- Het spaCy-model staat in `SPACY_MODEL`. Gebruik hetzelfde model als in de NLP-notebook (`nl_core_news_lg`), anders veranderen de lemma's en dus de resultaten.

In [ ]:
import nltk
import spacy

nltk.download("stopwords", quiet=True)
from nltk.corpus import stopwords

SPACY_MODEL = "nl_core_news_lg"           # zelfde model als in de NLP-notebook
STOP_WOORDEN = set(stopwords.words("dutch"))
nlp_model = spacy.load(SPACY_MODEL, disable=["parser", "ner"])   # parser/ner heb ik niet nodig

def verwijder_stopwoorden(zin):
    return " ".join(w for w in zin.split() if w.lower() not in STOP_WOORDEN)

def lemmatiseer(zin):
    return " ".join(t.lemma_ for t in nlp_model(zin) if not t.is_punct and not t.is_space)

@lru_cache(maxsize=None)
def _preprocess_tekst(tekst):
    tekst = tekst.lower()
    tekst = re.sub(r"[^\w\s]", " ", tekst)     # leestekens -> spatie (geen samengeplakte woorden)
    tekst = re.sub(r"\b\d+\b", " ", tekst)     # losse cijfers eruit
    tekst = " ".join(tekst.split())
    return lemmatiseer(verwijder_stopwoorden(tekst)).lower()

def preprocess(tekst):
    """Eén functie voor catalogusteksten én profielteksten."""
    return _preprocess_tekst("" if pd.isna(tekst) else str(tekst))

for naam in VARIANTEN:
    df[f"nlp_{naam}"] = df[f"tekst_{naam}"].map(preprocess)

df[["Titel", "tekst_fb", "nlp_fb"]].head(3)

## 4. Eén vaste split, vaste labels en geen lekkage

**Feedback:**
- Maak één vaste split op student-materiaalparen, met vaste labels uit `beoordeling` (bijvoorbeeld ≥ 4), en gebruik die voor alle modellen.
- Gebruik ontwikkeldata voor tuning en een onaangeraakte eindtest.
- Bouw per split de interactiescore alleen uit raw-interacties van de toegestane trainparen (merge op beide sleutels) en sluit alle validatie- en eindtestparen uit, ook bij de positieve geschiedenis in CB.
- Laat ontbrekende interacties NaN. De samengestelde score is alleen modelinput, geen evaluatiedoel.

**Toegepast:**
- Ik gebruik de `holdout`-functie uit de feedback (20% per student, minimaal 1 paar). Eerst wordt de **eindtest** afgesplitst, daarna uit de rest de **validatieset**. Het label is `relevant = gemiddelde beoordeling ≥ 4` per student-materiaalpaar. Dit zijn vaste labels die niet meer veranderen als ik de interactiescore aanpas.
- Voor de modelinput (geziene items en positieve geschiedenis) maak ik per situatie een `geschiedenis`. Die bestaat uit de raw-interacties zonder de achtergehouden paren. De uitsluiting doe ik met een merge op `student_id` én `cataloguscode`. De interactiescore wordt hierbij opnieuw berekend met `.map()` in plaats van een mogelijk muterende Series-view, en ontbrekende waarden blijven NaN.
- Een controle laat zien dat er geen enkel achtergehouden paar in de geschiedenis terechtkomt.

In [ ]:
# --- Vaste labels uit explicit beoordeling (herhaalde paren worden hier bewust geaggregeerd) ---
RELEVANT_VANAF = 4
labels = raw.groupby(["student_id", "cataloguscode"], as_index=False)["beoordeling"].mean()
labels = labels.dropna(subset=["beoordeling"]).reset_index(drop=True)
labels["relevant"] = labels["beoordeling"].ge(RELEVANT_VANAF)

def holdout(frame, seed):
    idx = []
    for _, g in frame.groupby("student_id"):
        if len(g) >= 2:
            idx.extend(g.sample(n=min(len(g) - 1, max(1, round(.2 * len(g)))), random_state=seed).index)
    return frame.drop(index=idx), frame.loc[idx]

ontwikkeling, eindtest = holdout(labels, 42)
train_labels, validatie = holdout(ontwikkeling, 17)

assert set(train_labels.index).isdisjoint(eindtest.index)
assert set(validatie.index).isdisjoint(eindtest.index)
assert set(train_labels.index).isdisjoint(validatie.index)

print(f"Gelabelde paren: {len(labels)} | train {len(train_labels)} | validatie {len(validatie)} | eindtest {len(eindtest)}")
print(f"Aandeel relevant (beoordeling >= {RELEVANT_VANAF}): {labels['relevant'].mean():.1%}")

In [ ]:
# --- Interactiescore: alleen modelinput. Zelfde recept als eerder, maar met .map() en NaN waar data ontbreekt ---
TYPE_SCORE = {"geliket": 5, "opgeslagen": 4, "gelezen": 3, "bekeken": 2, "niet_afgemaakt": 1, "afgewezen": 1}
GEWICHTEN = pd.Series({"leesprc": 0.15, "inttype": 0.35, "beoordeling": 0.50})

def bereken_intscore(frame):
    t = frame["interactietype"].astype(str).str.strip().str.casefold().map(TYPE_SCORE)
    t = t.fillna(pd.to_numeric(frame["interactietype"], errors="coerce"))   # al-omgezette cijfers blijven werken
    l = np.ceil(pd.to_numeric(frame["leespercentage"], errors="coerce") / 20).clip(1, 5)
    b = pd.to_numeric(frame["beoordeling"], errors="coerce")
    s = pd.DataFrame({"leesprc": l, "inttype": t, "beoordeling": b})
    gewogen = s.mul(GEWICHTEN, axis="columns").sum(axis=1, min_count=1)
    beschikbaar = s.notna().mul(GEWICHTEN, axis="columns").sum(axis=1)
    return gewogen / beschikbaar            # ontbrekend blijft NaN, geen nul

POSITIEF_VANAF = 3.5                        # intscore-drempel voor "leuk gevonden" als MODELINPUT

def bouw_geschiedenis(raw, uitsluiten, drempel=POSITIEF_VANAF):
    """Geschiedenis per student uit alleen de toegestane trainparen.
    Alle paren in `uitsluiten` (validatie/eindtest) worden via een merge op beide sleutels verwijderd."""
    uit = uitsluiten[["student_id", "cataloguscode"]].drop_duplicates().assign(_uit=1)
    train = raw.merge(uit, on=["student_id", "cataloguscode"], how="left")
    train = train[train["_uit"].isna()].drop(columns="_uit").copy()
    train["intscore"] = bereken_intscore(train)
    per_paar = train.groupby(["student_id", "cataloguscode"], as_index=False)["intscore"].mean()
    hist = {
        sid: {"gezien": set(g["cataloguscode"]),
              "positief": g.loc[g["intscore"] >= drempel, "cataloguscode"].tolist()}
        for sid, g in per_paar.groupby("student_id")
    }
    return hist, per_paar

def controleer_lekkage(hist, achtergehouden):
    lek = sum(c in hist.get(s, {}).get("gezien", set())
              for s, c in zip(achtergehouden["student_id"], achtergehouden["cataloguscode"]))
    assert lek == 0, f"{lek} achtergehouden paren zitten in de geschiedenis!"

# Ontwikkelfase: validatie én eindtest zijn verborgen
hist_dev, paren_dev = bouw_geschiedenis(raw, uitsluiten=pd.concat([validatie, eindtest]))
controleer_lekkage(hist_dev, validatie)
controleer_lekkage(hist_dev, eindtest)

# Eindfase: alleen de eindtest is verborgen (validatie hoort dan bij de training)
hist_final, paren_final = bouw_geschiedenis(raw, uitsluiten=eindtest)
controleer_lekkage(hist_final, eindtest)

# Productie (eindprototype): alle interacties zijn bekend
hist_all, paren_all = bouw_geschiedenis(raw, uitsluiten=labels.iloc[0:0])
print("Lekkagecontroles geslaagd.")

## 5. Personalisatie en filters: eisen versus voorkeuren

**Feedback:**
- Goed dat de positieve leesgeschiedenis wordt meegenomen en geziene items worden uitgesloten.
- Maak onderscheid tussen **eisen** (harde filters) en **voorkeuren** (zachte bonus). Gebruik niveau niet tegelijk als tekstkenmerk, bonus en harde filter.
- Taal is alleen nuttig als er variatie is of een gebruiker een andere taal vereist; meld dan dat er geen aanbod is.
- Gebruik `gegevensstatus` niet als beschikbaarheid.
- Controleer welke niveaus werkelijk voorkomen, inclusief combinaties zoals 2F/3F, en leg de compatibiliteitsregel vast.
- Behandel lege kandidaten, een nulvector en een onbekende student.
- Profieltekst = interesses (en eventueel genre), met `preprocess`. Genre en type alleen als zachte bonus.
- Pas dezelfde harde voorwaarden toe bij alle modellen. Geef bij een geschatte leestijd aan dat het een schatting is.

**Toegepast:**
- **Niveau** is nu alléén nog een harde eis (en staat niet meer in de tekst of als bonus). De regel staat in `niveau_compatibel`: een boek past als het niveau van de student en dat van het boek minstens één niveau gemeen hebben. Een boek met `2F/3F` past dus bij 2F én bij 3F. De cel hieronder laat zien welke niveaus echt voorkomen, zodat je kunt controleren of de regel klopt.
- **Leestijd** (`max_minuten`) en **taal** (`taal`) zijn optionele harde eisen. In de data is taal overal `NL`, dus een taalfilter verandert niets, tenzij iemand een andere taal vraagt. Dan meldt het model dat er geen aanbod is.
- **Genre en materiaaltype** zijn zachte bonussen (ze sluiten niets uit). De weging stem ik af op de validatieset (sectie 7).
- `gegevensstatus` gebruik ik nergens.
- Randgevallen geven een lege tabel met een uitleg in `attrs["opmerking"]`, in plaats van een foutmelding: onbekende student zonder voorkeuren, geen kandidaten na filteren, en een nulvector.

In [ ]:
# --- Welke niveaus komen echt voor? (basis voor de compatibiliteitsregel) ---
print("Niveaus in catalogus:\n", df["Niveau"].value_counts(dropna=False).to_string(), "\n")
print("Leesniveaus van studenten:\n", studenten["leesniveau"].value_counts(dropna=False).to_string(), "\n")
print("Talen in catalogus:", df["taal"].unique().tolist() if "taal" in df else "kolom ontbreekt",
      "-> taalfilter heeft alleen zin als hier meer dan één taal staat")

def niveau_tokens(x):
    """'2F/3F' -> {'2F','3F'}; '3F+' blijft '3F+'. Leeg/onbekend -> lege set."""
    if pd.isna(x):
        return frozenset()
    return frozenset(t for t in re.split(r"[\s/,;-]+", str(x).strip().upper()) if t)

def niveau_compatibel(student_niveau, boek_tokens):
    """REGEL: een boek past als student en boek minstens één niveau gemeen hebben.
    Is het studentniveau onbekend, dan is er geen eis. Is het boekniveau onbekend terwijl er wel
    een eis is, dan valt het boek af."""
    s = niveau_tokens(student_niveau)
    return (not s) or bool(s & boek_tokens)

# Aanbod per studentniveau (laat meteen zien of er overal genoeg aanbod is)
boek_tokens = df["Niveau"].map(niveau_tokens)
aanbod = {n: int(boek_tokens.map(lambda b, n=n: niveau_compatibel(n, b)).sum())
          for n in studenten["leesniveau"].dropna().unique()}
print("Aantal passende materialen per studentniveau:", aanbod)

In [ ]:
PROFIEL_VELDEN = ["interesses", "voorkeur_genre", "leesniveau", "voorkeur_materiaaltype"]
GEEN_VOORKEUR = {"", "nan", "none", "geen voorkeur", "gemengd"}
UITVOER_KOLOMMEN = ["cataloguscode", "Titel", "Niveau", "Type materiaal", "genre",
                    "geschatte_leestijd_min", "leestijd_geschat"]

def schoon_voorkeur(x):
    x = "" if pd.isna(x) else str(x).strip()
    return "" if x.casefold() in GEEN_VOORKEUR else x

def norm(serie):
    return serie.fillna("").astype(str).str.strip().str.casefold()

class ContentBasedRecommender:
    """TF-IDF + cosine similarity. Harde eisen: niveau, (taal), (max. leestijd).
    Zachte voorkeuren: genre en materiaaltype (bonus)."""

    def __init__(self, catalogus, studenten, tekstkolom, profiel_met_genre=True,
                 bonus_genre=0.15, bonus_type=0.15, profiel_gewicht=2):
        self.cat = catalogus.reset_index(drop=True)
        self.tekstkolom = tekstkolom
        self.profiel_met_genre = profiel_met_genre
        self.bonus_genre, self.bonus_type = bonus_genre, bonus_type
        self.profiel_gewicht = profiel_gewicht

        self.codes = self.cat["cataloguscode"].astype(str).to_numpy()
        self.pos = {c: i for i, c in enumerate(self.codes)}
        self.vectorizer = TfidfVectorizer()                      # opnieuw gefit op DEZE catalogustekst
        self.X = self.vectorizer.fit_transform(self.cat[tekstkolom].fillna("").astype(str))
        self.termen = self.vectorizer.get_feature_names_out()

        self.studenten = studenten.set_index("student_id")
        self._niveau = self.cat["Niveau"].map(niveau_tokens).to_numpy()
        self._genre = norm(self.cat["genre"]).to_numpy()
        self._type = norm(self.cat["Type materiaal"]).to_numpy()
        self._tijd = pd.to_numeric(self.cat["geschatte_leestijd_min"], errors="coerce").to_numpy()
        self._taal = norm(self.cat["taal"]).to_numpy() if "taal" in self.cat else None
        self._tijd_geschat = self.cat["leestijd_geschat"].fillna(False).astype(bool).to_numpy() \
            if "leestijd_geschat" in self.cat else np.zeros(len(self.cat), dtype=bool)

    # ---------- profiel ----------
    def _profiel(self, student_id, voorkeuren=None):
        if student_id in self.studenten.index:
            rij = self.studenten.loc[student_id]
            profiel = {k: schoon_voorkeur(rij.get(k)) for k in PROFIEL_VELDEN}
        elif voorkeuren:
            profiel = {k: "" for k in PROFIEL_VELDEN}
        else:
            return None                                        # onbekende student zonder voorkeuren
        for k, v in (voorkeuren or {}).items():                # opgegeven voorkeuren hebben voorrang
            if k in PROFIEL_VELDEN:
                profiel[k] = schoon_voorkeur(v)
        return profiel

    # ---------- harde eisen ----------
    def toegestane(self, student_id, geschiedenis=None, kandidaten=None, filters=True,
                   max_minuten=None, taal=None, voorkeuren=None, profiel=None):
        """Indices van toegestane kandidaten + opmerkingen. Geziene items vallen altijd af."""
        student_id = str(student_id)
        opm = []
        if profiel is None:
            profiel = self._profiel(student_id, voorkeuren)
        leeg = np.array([], dtype=int)
        if profiel is None:
            return leeg, [f"Onbekende student '{student_id}' en geen voorkeuren opgegeven."], None

        idx = np.arange(len(self.codes)) if kandidaten is None else \
            np.array([self.pos[c] for c in kandidaten if c in self.pos], dtype=int)
        gezien = (geschiedenis or {}).get(student_id, {}).get("gezien", set())
        if len(idx) and gezien:
            idx = idx[~np.isin(self.codes[idx], list(gezien))]
        if len(idx) == 0:
            return leeg, ["Geen kandidaten over na uitsluiten van reeds geziene materialen."], profiel

        if filters:
            if profiel["leesniveau"]:
                keep = np.array([niveau_compatibel(profiel["leesniveau"], self._niveau[i]) for i in idx])
                idx = idx[keep]
                if len(idx) == 0:
                    return leeg, [f"Geen aanbod op niveau '{profiel['leesniveau']}' (of een compatibel niveau)."], profiel
            else:
                opm.append("Geen leesniveau bekend: niveaufilter niet toegepast.")
            if taal and self._taal is not None:
                idx = idx[self._taal[idx] == str(taal).strip().casefold()]
                if len(idx) == 0:
                    return leeg, [f"Geen aanbod in de taal '{taal}'."], profiel
            if max_minuten is not None:
                tijd = self._tijd[idx]
                idx = idx[~np.isnan(tijd) & (tijd <= max_minuten)]
                if len(idx) == 0:
                    return leeg, [f"Geen materiaal met een (geschatte) leestijd van hooguit {max_minuten} minuten."], profiel
        return idx, opm, profiel

    def is_geschikt(self, student_id, code):
        """Voldoet het item aan de harde niveau-eis? (voor de geschiktheidsmaat zonder filters)"""
        profiel = self._profiel(str(student_id))
        return True if profiel is None or not profiel["leesniveau"] \
            else niveau_compatibel(profiel["leesniveau"], self._niveau[self.pos[code]])

    # ---------- aanbevelen ----------
    def recommend(self, student_id, aantal=3, geschiedenis=None, kandidaten=None, filters=True,
                  max_minuten=None, taal=None, voorkeuren=None, uitleg=False):
        student_id = str(student_id)
        idx, opm, profiel = self.toegestane(student_id, geschiedenis, kandidaten, filters,
                                            max_minuten, taal, voorkeuren)
        if len(idx) == 0:
            return self._leeg(opm)

        # profieltekst: interesses (+ genre) door DEZELFDE preprocess + tekst van positief beoordeelde items
        delen = [profiel["interesses"]] + ([profiel["voorkeur_genre"]] if self.profiel_met_genre else [])
        schoon = preprocess(" ".join(d for d in delen if d))
        positief = (geschiedenis or {}).get(student_id, {}).get("positief", [])
        pos_idx = [self.pos[c] for c in positief if c in self.pos]
        pos_tekst = " ".join(self.cat[self.tekstkolom].iloc[pos_idx].fillna("").astype(str)) if pos_idx else ""
        vec = self.vectorizer.transform([" ".join([schoon] * self.profiel_gewicht + [pos_tekst]).strip()])

        # zachte voorkeuren (bonus)
        genre_voorkeur, type_voorkeur = profiel["voorkeur_genre"].casefold(), profiel["voorkeur_materiaaltype"].casefold()
        bonus = np.zeros(len(idx))
        if genre_voorkeur:
            bonus += self.bonus_genre * (self._genre[idx] == genre_voorkeur)
        if type_voorkeur:
            bonus += self.bonus_type * (self._type[idx] == type_voorkeur)

        if vec.nnz == 0:                                       # nulvector
            if not (genre_voorkeur or type_voorkeur):
                return self._leeg(opm + ["Profiel bevat geen termen uit de catalogus en er zijn geen genre-/typevoorkeuren: "
                                         "te weinig informatie om te rangschikken."])
            opm.append("Geen overlap tussen profiel en catalogustermen: ranking alleen op genre/type-voorkeur.")
            sim = np.zeros(len(idx))
        else:
            sim = (self.X[idx] @ vec.T).toarray().ravel()

        score = sim + bonus
        volgorde = np.argsort(-score, kind="stable")[:aantal]
        sel = idx[volgorde]
        res = self.cat.iloc[sel][[c for c in UITVOER_KOLOMMEN if c in self.cat]].copy().reset_index(drop=True)
        res["tekst_similarity"] = sim[volgorde]
        res["match_score"] = score[volgorde]
        if uitleg:
            res["uitleg"] = [self._uitleg(vec, i, profiel, bool(pos_idx)) for i in sel]
        res.attrs["opmerking"] = " | ".join(opm)
        return res

    def _uitleg(self, vec, i, profiel, heeft_historie=False):
        delen = []
        if vec.nnz:
            bijdrage = vec.multiply(self.X[i]).toarray().ravel()
            woorden = [self.termen[t] for t in np.argsort(-bijdrage)[:3] if bijdrage[t] > 0]
            if woorden:
                delen.append(("sluit aan bij je interesses en leesgeschiedenis: " if heeft_historie else "sluit aan bij je interesses: ")
                             + ", ".join(woorden))
        if profiel["voorkeur_genre"] and self._genre[i] == profiel["voorkeur_genre"].casefold():
            delen.append(f"genre {self.cat.at[i, 'genre']} (jouw voorkeur)")
        if profiel["voorkeur_materiaaltype"] and self._type[i] == profiel["voorkeur_materiaaltype"].casefold():
            delen.append(f"type {self.cat.at[i, 'Type materiaal']} (jouw voorkeur)")
        if profiel["leesniveau"]:
            delen.append(f"niveau {self.cat.at[i, 'Niveau']} past bij {profiel['leesniveau']}")
        if self._tijd_geschat[i]:
            delen.append("let op: de leestijd is een schatting")
        return "; ".join(delen) or "geen specifieke match gevonden"

    def _leeg(self, opmerkingen):
        res = pd.DataFrame(columns=[c for c in UITVOER_KOLOMMEN if c in self.cat] + ["tekst_similarity", "match_score"])
        res.attrs["opmerking"] = " | ".join(opmerkingen)
        return res

## 6. Evaluatie: één protocol, twee methodes apart, met baselines

**Feedback:**
- Het onderscheid tussen full-catalog en candidate ranking is nuttig, maar candidate ranking meet alleen rangschikking binnen *bekende testitems*. Onbeoordeeld betekent onbekend, niet afgewezen. Rapporteer beide protocollen apart.
- CB gebruikte drempel 3.5 en één split, CF drempel 4 en tien splits. Vergelijk niet met cijfers uit verschillende protocollen.
- Gebruik één gedeelde metricfunctie, dezelfde kandidaten, K en gebruikers; rapporteer ook het aantal uitgesloten studenten zonder relevante items en de dekking.
- Evalueer de ranking zonder filters als aparte diagnose, maar rapporteer ook geschiktheid en dekking mét filters.
- Maak één tabel met baseline en modellen op dezelfde eindtest; noem de full-catalog-metrics *observed-hit metrics*.

**Toegepast:**
- `ranking_metrics` komt uit de feedback en geldt voor alle modellen (Precision, Recall, F1, NDCG).
- `evalueer` voert een model uit op de vaste labels, voor beide protocollen:
  - **candidate ranking:** het model rangschikt per student alleen diens achtergehouden testitems;
  - **full-catalog (observed-hit):** het model kiest uit de hele catalogus minus wat de student al zag. Een "hit" telt alleen als het item een bekend, relevant testitem is.
- Elke run rapporteert het aantal studenten, het aantal uitgesloten studenten (geen relevant item in de testset), de **dekking** (aandeel studenten met genoeg aanbevelingen) en de **geschiktheid** (aandeel aanbevelingen dat aan de niveau-eis voldoet).
- Als baselines gebruik ik **random** (gemiddelde over 20 seeds) en **populariteit** (aantal positieve train-interacties). Ze krijgen exact dezelfde kandidaten en filters als het CB-model.
- Een model krijgt een `rank_fn(student_id, kandidaten) -> lijst met codes`. Zo kunnen de CF-modellen er later zonder aanpassing in.

In [ ]:
TOP_K = 3

def ranking_metrics(ranking, relevant, k=3):
    ranking = list(dict.fromkeys(ranking))[:k]
    relevant = set(relevant)
    hits = np.array([x in relevant for x in ranking], dtype=float)
    p = hits.sum() / k
    r = hits.sum() / len(relevant) if relevant else np.nan
    dcg = (hits / np.log2(np.arange(2, len(hits) + 2))).sum()
    idcg = (1 / np.log2(np.arange(2, min(k, len(relevant)) + 2))).sum()
    return {"Precision": p, "Recall": r,
            "F1": 2 * p * r / (p + r) if p + r > 0 else (0 if relevant else np.nan),
            "NDCG": dcg / idcg if idcg else np.nan,
            "Aantal aanbevelingen": len(ranking)}

def evalueer(rank_fn, test, protocol, k=TOP_K, geschikt_fn=None):
    """protocol: 'candidate' (rangschik alleen de testitems van de student) of 'full' (hele catalogus)."""
    rijen, uitgesloten = [], 0
    for sid, g in test.groupby("student_id"):
        relevant = set(g.loc[g["relevant"], "cataloguscode"])
        if not relevant:
            uitgesloten += 1                      # student heeft geen relevant item in deze testset
            continue
        kandidaten = g["cataloguscode"].tolist() if protocol == "candidate" else None
        ranking = list(dict.fromkeys(rank_fn(sid, kandidaten)))[:k]
        rij = {"student_id": sid, **ranking_metrics(ranking, relevant, k)}
        if geschikt_fn is not None and ranking:
            rij["Geschiktheid"] = np.mean([geschikt_fn(sid, c) for c in ranking])
        rijen.append(rij)
    res = pd.DataFrame(rijen)
    return {"studenten": len(res), "uitgesloten": uitgesloten,
            f"Precision@{k}": res["Precision"].mean(), f"Recall@{k}": res["Recall"].mean(),
            f"NDCG@{k}": res["NDCG"].mean(),
            "dekking": (res["Aantal aanbevelingen"] >= k).mean(),
            "geschiktheid": res["Geschiktheid"].mean() if "Geschiktheid" in res else np.nan}

# ---- rank-functies: elk model/baseline krijgt dezelfde kandidaten en dezelfde filters ----
def rankfn_cb(model, hist, filters, k=TOP_K):
    return lambda sid, kand: model.recommend(sid, k, hist, kand, filters=filters)["cataloguscode"].tolist()

def rankfn_populariteit(model, per_paar, hist, filters, k=TOP_K):
    pop = per_paar.assign(pos=per_paar["intscore"] >= POSITIEF_VANAF).groupby("cataloguscode") \
                  .agg(n_pos=("pos", "sum"), gem=("intscore", "mean"))
    score = pd.Series(0.0, index=model.codes)
    gem_idx = pop.index.intersection(score.index)
    score.loc[gem_idx] = pop.loc[gem_idx, "n_pos"] + pop.loc[gem_idx, "gem"].fillna(0) / 10   # gem = tiebreak
    score = score.to_numpy()
    def rank(sid, kand):
        idx, _, _ = model.toegestane(sid, hist, kand, filters)
        return model.codes[idx[np.argsort(-score[idx], kind="stable")[:k]]].tolist()
    return rank

def rankfn_random(model, hist, filters, seed, k=TOP_K):
    rng = np.random.default_rng(seed)
    def rank(sid, kand):
        idx, _, _ = model.toegestane(sid, hist, kand, filters)
        return model.codes[rng.permutation(idx)[:k]].tolist()
    return rank

def evalueer_random(model, hist, test, protocol, filters, n_seeds=20, k=TOP_K):
    runs = [evalueer(rankfn_random(model, hist, filters, s, k), test, protocol, k, model.is_geschikt)
            for s in range(n_seeds)]
    return pd.DataFrame(runs).mean().to_dict()

## 7. Tuning op validatie, daarna één keer op de eindtest

**Feedback:**
- Tune op validatie, train de gekozen instellingen opnieuw op ontwikkeling en evalueer één keer op de eindtest. Gebruik dezelfde splits niet tegelijk voor selectie en eindrapportage.
- Geen extra uitgebreide search nodig.
- Onderzoek de dubbele weging van onderwerp en vergelijk de tekstselectie met de bestaande variant.

**Toegepast:**
- Ik doe een kleine grid: 4 tekstvarianten × 3 bonuswegingen voor genre/type = 12 configuraties, alleen op de **validatieset** (geschiedenis zonder validatie en eindtest).
- Selectiecriterium: **NDCG@3 bij candidate ranking zonder filters**. Dat meet de kwaliteit van de ranking zelf, zonder dat de filters het aantal aanbevelingen beïnvloeden. Die keuze zit in `SELECTIE`, zodat je hem kunt aanpassen. Beide protocollen staan wel in de tabel.
- De eindtest blijft tot sectie 8 onaangeraakt.

In [ ]:
BONUS_GRID = [(0.0, 0.0), (0.05, 0.05), (0.15, 0.15)]     # (bonus_genre, bonus_type)
SELECTIE = ("candidate", f"NDCG@{TOP_K}")                  # hiermee kies ik de beste configuratie

def bouw_model(variant, bonus_genre, bonus_type):
    return ContentBasedRecommender(df, studenten, tekstkolom=f"nlp_{variant}",
                                   profiel_met_genre="genre" in VARIANTEN[variant],
                                   bonus_genre=bonus_genre, bonus_type=bonus_type)

rijen = []
for variant in VARIANTEN:
    for bg, bt in BONUS_GRID:
        model = bouw_model(variant, bg, bt)
        for protocol in ("candidate", "full"):
            res = evalueer(rankfn_cb(model, hist_dev, filters=False), validatie, protocol,
                           geschikt_fn=model.is_geschikt)
            rijen.append({"variant": variant, "bonus_genre": bg, "bonus_type": bt, "protocol": protocol, **res})

tuning = pd.DataFrame(rijen)
tabel = tuning.pivot_table(index=["variant", "bonus_genre", "bonus_type"], columns="protocol",
                           values=[f"Precision@{TOP_K}", f"NDCG@{TOP_K}"]).round(3)
display(tabel)

beste_rij = tuning[tuning["protocol"] == SELECTIE[0]].sort_values(SELECTIE[1], ascending=False).iloc[0]
BESTE = {"variant": str(beste_rij["variant"]), "bonus_genre": float(beste_rij["bonus_genre"]),
         "bonus_type": float(beste_rij["bonus_type"])}
print("Gekozen op validatie:", BESTE)

**Interpretatie (vul in na het draaien):**
- Welke tekstvariant scoort het best, en is het verschil groot of binnen de ruis? Let op: de validatieset is klein, dus kleine verschillen zijn niet betekenisvol.
- Helpt `fb_plus_onderwerp` (dubbele weging van onderwerp) of niet?
- Helpt de zachte bonus voor genre/type, of beter 0?
- Wat is de verhouding tot de oude variant (`oud`)? Dat is de vergelijking die de docent vroeg onder hetzelfde protocol.

### Eindtest (één keer)

**Feedback:** evalueer één keer op de eindtest en maak één tabel met baseline en modellen. Noem de full-catalog-metrics observed-hit metrics. Bewaar oude resultaten als referentie.

**Toegepast:**
- Het gekozen model train ik opnieuw op de volledige ontwikkelingsdata. De geschiedenis bevat dus nu ook de validatieparen en alleen de eindtest blijft verborgen.
- De eindtabel bevat random, populariteit en het gekozen CB-model, elk voor beide protocollen en met en zonder filters.
- De oude tekstselectie staat er **ter referentie** bij. Hij is niet gebruikt voor de selectie. Zo wordt de eindtest niet gebruikt om te kiezen.
- De uitkomst print ik per protocol en filterstand, alleen met Precision@3 en NDCG@3 per model. Dezelfde `evalueer()`-functie kan later ook voor de CF-modellen gebruikt worden, zodat die onder dezelfde eindtest naast CB komen te staan.

In [ ]:
m_beste = bouw_model(**BESTE)
m_oud   = bouw_model("oud", BESTE["bonus_genre"], BESTE["bonus_type"])     # alleen ter referentie

PROTOCOLLEN = {"candidate": "candidate ranking", "full": "full-catalog (observed-hit)"}
rijen = []
for protocol, protocol_naam in PROTOCOLLEN.items():
    for filters in (False, True):
        modellen = {
            "random (gem. 20 seeds)": lambda: evalueer_random(m_beste, hist_final, eindtest, protocol, filters),
            "populariteit":           lambda: evalueer(rankfn_populariteit(m_beste, paren_final, hist_final, filters),
                                                       eindtest, protocol, geschikt_fn=m_beste.is_geschikt),
            f"CB ({BESTE['variant']})": lambda: evalueer(rankfn_cb(m_beste, hist_final, filters), eindtest, protocol,
                                                          geschikt_fn=m_beste.is_geschikt),
        }
        if BESTE["variant"] != "oud":     # de oude tekst alleen ter referentie, niet gebruikt voor selectie
            modellen["CB oude tekst (referentie)"] = lambda: evalueer(rankfn_cb(m_oud, hist_final, filters), eindtest,
                                                                       protocol, geschikt_fn=m_oud.is_geschikt)
        for naam, run in modellen.items():
            rijen.append({"protocol": protocol_naam, "filters": "met" if filters else "zonder", "model": naam, **run()})

eindtabel = pd.DataFrame(rijen).set_index(["protocol", "filters", "model"])

for (protocol, filters), blok in eindtabel.groupby(level=["protocol", "filters"], sort=False):
    print(f"\n{protocol} - {filters} filters")
    for (_, _, model), r in blok.iterrows():
        print(f"  {model}: Precision@{TOP_K} = {r[f'Precision@{TOP_K}']:.3f} | NDCG@{TOP_K} = {r[f'NDCG@{TOP_K}']:.3f}")

**Interpretatie (vul in na het draaien):**
- Verslaat CB de populariteits- en random-baseline? Zo niet, benoem dat gewoon eerlijk.
- Wat gebeurt er met de scores en de dekking als de filters aan staan? Een lagere score met filters kan ook komen doordat er minder kandidaten overblijven, niet alleen door een slechter model. Bij candidate ranking zijn dat er per student maar een paar, dus na het niveaufilter blijven er vaak minder dan K = 3 over. Dat zie je terug in een lage dekking. Vergelijk de modellen daarom onderling en lees de lagere absolute score niet als een slechter model.
- Hoe laag is de **geschiktheid** zonder filters? Dat laat zien waarom de harde niveau-eis nodig is.
- De full-catalog-cijfers zijn *observed-hit metrics*: een goede aanbeveling die de student nog nooit las, telt niet mee. Ze zijn dus een ondergrens.

### Evaluatiemethode: candidate ranking versus full-catalog (aangepaste tekst)

*Dit vervangt de eerdere uitleg uit de notebook. De eerdere conclusie ("toont zuiver aan hoe goed het algoritme onderscheid maakt") was te stellig.*

- **Candidate ranking** laat het model per student alleen de achtergehouden testitems rangschikken. Dat meet of het model binnen *bekende* items relevante boven niet-relevante zet. Het zegt niets over de kwaliteit van aanbevelingen uit de hele catalogus.
- **Full-catalog ranking** kiest uit alle nog ongeziene materialen. Omdat slechts ca. 7% van alle student-materiaalcombinaties beoordeeld is, zijn dit *observed-hit metrics*: onbeoordeeld betekent *onbekend*, niet *afgewezen*. Een goed passend boek dat nog nooit door die student is gelezen, telt als "fout". De scores zijn dus laag en vooral bruikbaar om modellen onderling te vergelijken.
- Beide protocollen worden **apart** gerapporteerd. Voor een echte validatie zou een kleine groep studenten een blind samengestelde pool van aanbevelingen moeten beoordelen.

## 8. Eindprototype: één gebruikersfunctie met uitleg en tests

**Feedback:** maak één gebruikersfunctie die `student_id`, optionele voorkeuren en `top_k` accepteert, de gekozen methode gebruikt en titel, niveau, materiaaltype, score en een concrete uitleg teruggeeft. Test een bestaande student, een nieuwe student zonder historie, geen passende kandidaten, weinig buren en een maximale leestijd. Toon geen persoonsgegevens van vergelijkbare studenten. Leg uit dat scores geen kansen zijn.

**Toegepast:**
- `geef_aanbevelingen()` is de recommender zelf: standaard **3 boeken** (`top_k=3`). `toon_aanbevelingen()` laat per boek zien *waarom* het bij de student past.
- `geef_aanbevelingen()` gebruikt het gekozen CB-model met de volledige geschiedenis (alle bekende interacties) en altijd met filters.
- De uitleg noemt de woorden die de match veroorzaken, de voorkeuren waar het item aan voldoet, het niveau, en of de leestijd een schatting is. CB gebruikt geen gegevens van andere studenten, dus er worden ook geen persoonsgegevens van vergelijkbare studenten getoond.
- Het onderdeel "weinig buren" geldt alleen voor user-based CF en is bij CB niet van toepassing. Dat staat in de tests als opmerking.
- De `match_score` is een relatieve score voor sortering en geen kans of kwaliteitsoordeel.

In [ ]:
MODEL_FINAL = bouw_model(**BESTE)

def geef_aanbevelingen(student_id, top_k=3, voorkeuren=None, max_minuten=None, taal=None):
    """Aanbevelingen voor een student (of een nieuwe gebruiker met alleen `voorkeuren`).
    voorkeuren: dict met o.a. 'interesses', 'voorkeur_genre', 'leesniveau', 'voorkeur_materiaaltype'."""
    res = MODEL_FINAL.recommend(student_id, aantal=top_k, geschiedenis=hist_all, filters=True,
                                max_minuten=max_minuten, taal=taal, voorkeuren=voorkeuren, uitleg=True)
    kolommen = ["Titel", "Niveau", "Type materiaal", "geschatte_leestijd_min", "match_score", "uitleg"]
    uit = res[[c for c in kolommen if c in res.columns]].copy()
    uit.attrs["opmerking"] = res.attrs.get("opmerking", "")
    return uit

def toon_aanbevelingen(student_id, top_k=3, **kwargs):
    """Toont de aanbevolen boeken met per boek de reden ('Waarom'), zoals een gebruiker het zou zien."""
    res = geef_aanbevelingen(student_id, top_k=top_k, **kwargs)
    if res.attrs.get("opmerking"):
        print("Opmerking:", res.attrs["opmerking"])
    if res.empty:
        print("Geen aanbevelingen gevonden.")
        return
    print(f"Aanbevelingen voor {student_id}:\n")
    for n, (_, r) in enumerate(res.iterrows(), 1):
        print(f"{n}. {r['Titel']}  [{r['Niveau']}, {r['Type materiaal']}]")
        print(f"   Waarom: {r['uitleg']}\n")

def toon(titel, res):
    print(f"\n=== {titel} ===")
    if res.attrs.get("opmerking"):
        print("Opmerking:", res.attrs["opmerking"])
    display(res if len(res) else pd.DataFrame({"resultaat": ["(geen aanbevelingen)"]}))

print("Let op: match_score is een relatieve sorteerscore, geen kans of kwaliteitsoordeel.")

In [ ]:
# --- Tests voor de randgevallen uit de feedback ---
voorbeeld = studenten.dropna(subset=["leesniveau"]).iloc[0]

# --- De recommender zoals een gebruiker hem ziet: 3 boeken met per boek de reden ---
toon_aanbevelingen(voorbeeld["student_id"])

toon(f"1. Bestaande student ({voorbeeld['student_id']})",
     geef_aanbevelingen(voorbeeld["student_id"], top_k=3))

toon("2. Nieuwe student zonder historie (alleen voorkeuren)",
     geef_aanbevelingen("NIEUW-001", top_k=3,
                        voorkeuren={"interesses": voorbeeld["interesses"], "leesniveau": voorbeeld["leesniveau"]}))

toon("3a. Onbekende student zonder voorkeuren", geef_aanbevelingen("BESTAAT-NIET", top_k=3))

toon("3b. Geen passende kandidaten (niveau dat niet in de catalogus voorkomt)",
     geef_aanbevelingen("NIEUW-002", top_k=3,
                        voorkeuren={"interesses": voorbeeld["interesses"], "leesniveau": "9Z"}))

toon("4a. Maximale leestijd: 60 minuten",
     geef_aanbevelingen(voorbeeld["student_id"], top_k=3, max_minuten=60))

toon("4b. Maximale leestijd zo klein dat niets past (1 minuut)",
     geef_aanbevelingen(voorbeeld["student_id"], top_k=3, max_minuten=1))

toon("5. Nulvector: interesses die niet in de catalogus voorkomen, geen andere voorkeuren",
     geef_aanbevelingen("NIEUW-003", top_k=3, voorkeuren={"interesses": "qqqxyz wwwvvv", "leesniveau": voorbeeld["leesniveau"]}))

toon("6. Taalfilter op een taal zonder aanbod",
     geef_aanbevelingen(voorbeeld["student_id"], top_k=3, taal="EN"))

# 7. 'Weinig buren' hoort bij user-based CF. CB gebruikt geen andere studenten, dus niet van toepassing.

## 9. Beperkingen en conclusie

**Feedback:** rapporteer beperkingen en waarom het eindmodel passend is. Niet alle algoritmen hoeven te winnen. Bespreek populariteitsbias, beperkte of synthetische interactiedata, privacy en keuzevrijheid.

**Toegepast:** hieronder staat het stramien. De cijfers vul ik in na het draaien van sectie 7 en 8. Ik trek geen conclusies die de tabel niet ondersteunt.

**Conclusie (vul in):**
- Gekozen tekstselectie en wegingen: *…* (gekozen op validatie).
- Resultaat op de eindtest ten opzichte van random en populariteit, per protocol: *…*
- Effect van de filters op score, dekking en geschiktheid: *…*
- Waarom CB passend is voor het eindprototype: werkt ook voor nieuwe studenten zonder historie (cold start), is uitlegbaar via woorden en voorkeuren en gebruikt geen gegevens van andere studenten.

**Beperkingen:**
- **Weinig en deels synthetische data:** de interactiedata is beperkt (ca. 7% gevuld) en deels gesimuleerd. Kleine verschillen tussen varianten zijn niet betrouwbaar en de resultaten zeggen weinig over echte studenten.
- **Observed-hit metrics:** onbeoordeeld is onbekend, niet afgewezen. Full-catalog-scores zijn een ondergrens en candidate ranking meet alleen binnen bekende items. Voor echte validatie is een blinde beoordeling door gebruikers nodig.
- **Eén split, kleine testset:** de eindtest bevat per student maar enkele items. Studenten zonder relevant testitem zijn uitgesloten (zie de tabel).
- **Leestijd is deels een schatting** (uit het aantal pagina's of omgekeerd) en moet zo worden gepresenteerd.
- **Populariteitsbias en filterbubbel:** TF-IDF op interesses en eerder gelezen items beveelt vooral meer van hetzelfde aan. Dat beperkt ontdekking en keuzevrijheid. Een gebruiker moet kunnen afwijken van de aanbevelingen en voorkeuren kunnen aanpassen.
- **Scores zijn geen kansen:** `match_score` is een relatief sorteergetal.
- **Privacy:** CB gebruikt alleen het eigen profiel en de eigen geschiedenis van de student, en toont geen gegevens van anderen. Interactiegegevens blijven wel persoonsgegevens en moeten zo behandeld worden.